In [ ]:
import duckdb

# # Read-only (safe while no build is running)
# conn = duckdb.connect("data_manager/catalog_v2.duckdb", read_only=True)

# Read-only snapshot while a build IS running (copy first)
import shutil, os, time
snap = "/tmp/catalog_snap.duckdb"
for _ in range(60):
    try:
        shutil.copy2("../../data_manager/catalog_v2.duckdb", snap)
        conn = duckdb.connect(snap, read_only=True)
        break
    except duckdb.IOException:
        time.sleep(0.5)
# ... use conn ...


In [ ]:
from retriever import EventCatalog

with EventCatalog() as cat:
    # Summary dict: {(source, data_class): count}
    print(cat.summary())

    # Single event lookup
    ev = cat.get(source='exp_reco', season=2020, cluster=3, run='228', event_id=42)
    hits = ev.load_hits()     # (n_hits, 5) float32  [amp, t, x, y, z]
    reco = ev.load_reco()     # dict of reco scalars (exp_reco only)
    print(ev.info)

    # Batch query → list[Event]
    events = cat.query(source='exp', season=2020, cluster=7)
    hits_list = [e.load_hits() for e in events[:10]]

    # Batch query → pandas DataFrame (metadata only)
    df = cat.query_df(source='mc_merged', data_class='nuatm_2020')

In [ ]:
conn.close()
os.unlink(snap)